In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\powerplant_data.csv")

In [ ]:
df.head(5)
# AT - temperature value
# V - vaccum
# AO - Pressure
# RH - Humidity

# PE - Prouduce Energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [ ]:
y.head()

In [ ]:
# Training in Neural network
# 1. Load the Dataset
# 2. Data convert to Tensors
# 3. TensorDataset and DataLoader
# 4. Define ANN model  
# 5. Train the model (save the model too)
# 6. Evaluation

In [ ]:
# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1 ,1)
# y_train.values because, its a pandas core series not a numpy array
# .view(-1, 1) because its only has one column.

In [ ]:
# Data Loader - to define how the data will be loaded for training
# - creates batches for us
# - shuffles (optional )
# Tensor Dataset class - it recives raw data, and transfer it to Data Loader(DL cant access raw data from memory).
# - access data
# - can access one sample / one row(features, target)

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Deep Learning

In [ ]:
# Defined Artificial Neural Network Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(6, 6),
            nn.ReLU(),

            #output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)


# No need to write backward propagation.

In [ ]:
import torch.optim as optim

model = ANN()

# Loss, optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN

train_loss = []
val_losses = []

best_val_loss = float("inf")

epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # trainig loss for 1 epoch

    for xb, yb, in train_loader:
        #xb - features of 1 batch
        #yb - labels of 1 batch

        optimizer.zero_grad() # Fresh gradients repeatedly

        # 1. forward Propagation.
        outputs = model(xb) # Predicted outputs for this batch

        # 2. compute loss.
        loss = criterion(outputs, yb)

        # 3. backward propagation
        loss.backward()

        # 4. update parameters
        optimizer.step()

        # running loss - for all the batches.
        running_loss += loss.item() # convert tensor to float values with .items()
    
    epoch_train_loss  = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # so that gradient is not computed
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") # .pt or .pth
 

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training loss": train_loss,
    "Validation loss": val_losses 
})

plt.plot(loss_df["Training loss"], label = "Training Loss")
plt.plot(loss_df["Validation loss"], label = "Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Losses Values")
plt.legend()

In [ ]:
# Loading the best model
model.load_state_dict(torch.load("best_model.pt"))
# model contains value with best_val_loss

In [ ]:
# Evaluating the model
# Steps:
# Training MSE
# Testing MSE
# R2 - R2 Score

model.eval()
with torch.no_grad():
    train_predict = model(X_train_tensor)
    test_predict = model(X_test_tensor)

    train_mse_loss = criterion(train_predict, y_train_tensor)
    test_mse_loss = criterion(test_predict, y_test_tensor)

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print("R^2 Score: ", r2_score(y_test, test_predict))

In [ ]:
predicted_df = pd.DataFrame(test_predict.numpy(), columns=["Predicted Value"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)